In [ ]:
# Paths are resolved relative to the repository, or to $PUBO_DATA if set.
# The original research code used absolute paths on the authors' machine;
# only the root has been parameterised, the directory layout is unchanged.
import os
DATA_ROOT = os.environ.get('PUBO_DATA',
                           os.path.join(os.path.dirname(os.getcwd()), '..', 'data', 'legacy'))
EXTERNAL_ROOT = os.environ.get('PUBO_EXTERNAL', DATA_ROOT)  # data not shipped with the repo


In [ ]:
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import time
import pickle
import pandas as pd
from IPython.display import clear_output

In [ ]:
 # amplitude increment
class SIMCIM():
    #Initial parameters
    def __init__(self, J3, O,S,D, N,dt, tau, dim, sigma, zeta, alpha,  attempt_num,  device='cuda'):
        
        self.J3=torch.tensor(J3, device=device, dtype=torch.float)
        
        
        #Parameters of the algorithm (for adjusting)
        self.O=O
        self.S=S
        self.D=D
        self.N=N
        self.tau=tau
        self.zeta=zeta
        self.alpha=alpha
        self.zeta=zeta
        self.dt=dt
        self.sigma = sigma
        #Parameters of the algorithm (will not be changed)
        c_th=1
        self.attempt_num = attempt_num
        self.Jmax = torch.mean(torch.sum(torch.abs(self.J3),(1,2)))
        self.device = device
        self.dim=self.J3.size(0)
        self.tau=int(2*self.J3.size(0))
        #self.lastcuts=torch.zeros(batch)
    #amplitude increment
    
    def ampl_inc(self, c,p):
        return (p*c - self.zeta/self.Jmax*(3*torch.einsum('ijk,js,ks->is',(self.J3, c, c))))*self.dt + self.sigma*torch.randn_like(c)


    # pump ramp
    def pump(self):
        j = torch.arange(self.N, dtype = torch.float, device=self.device)
        arg = torch.tensor(self.S,dtype = torch.float, device=self.device)*(j/self.N-0.5)
        return self.Jmax/self.dim*self.O*(torch.tanh(arg) + self.D )
    

    
    
    # evolution of amplitudes
    # init_value -- init coupling
    # final_value -- final coupling
    # N -- number of time iterations
    # attempt_num -- number of runs 
    # J -- coupling matrix
    # b -- biases
    # O, S, D -- pump parameters
    # sigma -- sqrt(std) for random number
    # alpha -- momentum parameter
    # c_th -- restriction on amplitudes growth
    def evolve(self):
        
        #dimension of the arrays

        i=0
        #random
        random_attempt = np.random.randint(self.attempt_num)

        # initializing current amplitudes
        c = torch.zeros((self.dim, self.attempt_num),dtype=torch.float, device=self.device)
        dc_momentum = torch.zeros_like(c)
        spins_opt=torch.zeros_like(c)
        spins=torch.sign(c)
        dc=torch.zeros_like(c)

        # define pump array
        p = self.pump()
        H_opt=torch.einsum('ijk,is,js,ks->s',(self.J3, spins,spins, spins))
#         t1=0
        #initial values of amplitude and dc
        
        
        # initializing moving average of amplitudes increment
        
        #c_all=torch.zeros((dim, attempt_num, N), dtype=torch.float)
        for i in range(1,self.N):
            

            dc = self.ampl_inc(c, p[i])
#             zeta=self.coupling(t1,rate, device=self.device)
            
            # calculating moving average of amplitudes increment
            dc_momentum = self.alpha*dc_momentum + (1-self.alpha)*dc
            c1=c+dc_momentum
            check1=(torch.abs(c1)<=1.0).float()

            c=c*(1-check1)+check1*c1
            # calculating possible values of amplitudes on the next 
            spins=torch.sign(c)
            H = torch.einsum('ijk,is,js,ks->s',(self.J3, spins,spins, spins))
            check = (H<H_opt).type(torch.float)
            H_opt = check*H+(1-check)*H_opt
            spins_opt = check*spins + (1-check) * spins_opt

        spins = spins_opt
        self.lastcuts = H_opt 
       
        return self.lastcuts, spins_opt

In [ ]:
import mloop.interfaces as mli
import mloop.controllers as mlc
import mloop.visualizations as mlv

In [ ]:
class SimcimInterface(mli.Interface):
    def __init__(self, simcim, attempt_num, clip_percents): 
        super(SimcimInterface, self).__init__()
        self.simcim=simcim
        self.attempt_num = attempt_num
        self.simcim.evolve( )
        self.mean = np.mean(np.array(simcim.lastcuts.cpu()))
        self.scale = np.min(np.array(simcim.lastcuts.cpu()))#+self.perc
        self.minimum = np.min(np.array(simcim.lastcuts.cpu()))
        #self.simcim.pump(O,S,D,N, Jmax)

        #self.simcim.pump(O,S,D,N, Jmax)



    def get_sim_params(self, packed_params):
        num_single = 6
        logsig, logdt, logzeta, logS, logD, logO  = packed_params[:num_single]

        sigma = 10**logsig
        dt = 10**logdt
        zeta = 10**logzeta
        S=10**logS
        D=-10**logD
        O=10**logO

        return sigma, dt, zeta, S, D, O


    def get_next_cost_dict(self, params_dict):
        params = params_dict['params']

        sigma, dt, zeta, S, D, O = self.get_sim_params(params)

        print('\nSIMCIM:\nsigma={}\ndt={}\nzeta={}\nS={}\nD={}\nO={}\n'.format(sigma,dt,zeta, S, D, O))

        self.simcim.sigma = sigma
        self.simcim.dt = dt
        self.simcim.zeta = zeta
        self.simcim.S=S
        self.simcim.O=O
        self.simcim.D=D
        #self.simcim.rate=rate
        self.simcim.evolve()
        cost_perc = min([np.mean(np.array(self.simcim.lastcuts.cpu()))/np.abs(self.mean),1]) #np.min(self.simcim.lastcuts)
        cost_min = min([np.min(np.array(self.simcim.lastcuts.cpu()))/np.abs(self.scale), 1]) 
        cost= cost_min + 0.25 * cost_perc
        #cost=min([np.min(self.simcim.lastcuts.cpu())/self.minimum, 1])
        uncer = 0
        bad = False
        print('Initial perc', self.mean)
        target_perc=self.mean
        target_scale=self.scale
        print('Final perc', np.mean(np.array(self.simcim.lastcuts.cpu())))
        print('Final energy', np.min(np.array(self.simcim.lastcuts.cpu())))
        cost_dict = {'cost':cost, 'uncer':uncer, 'bad':bad}
        return cost_dict

In [ ]:

alpha = 1-0.99


for i in range(9,10):
    psize=int(10*(i+1))
    for number in range(10):
        params= np.load(DATA_ROOT+'/sk_symmetric/PolySimCIM_each/'+str(psize)+'_'+str(number)+'_params.npy',)
        foldername = "sk_symmetric/"+str(psize)+'_'+str(number)
        K= torch.tensor(np.load(DATA_ROOT+'/'+foldername+'.npy'), dtype = torch.float, device=torch.device('cuda'))
    #K = K_imported.reshape(100,100,100)
        unity=torch.ones((K.size(0)), dtype=torch.float32).cuda()
        dim=K.size(0)
        sigma, dt, zeta, S, D, O = np.array(params)# + np.array(params) * np.random.randn(6)

        tau=dim
        N = 1000
        attempt_num = 2000
        
        Jmax = torch.max(torch.sum(torch.abs(K),1))

        sim=SIMCIM(K, O,S,D, N,dt, tau, dim, sigma, zeta, alpha, attempt_num,  device='cuda')
        H_opt, c =sim.evolve()
#         if number==0:
#             logparams=controller.best_params
        logparams = np.log10(np.array([sigma, dt, zeta, S, np.abs(D), O]))
        min_boundary = np.array([-3,-4,-4,-4,-4,-4])
        max_boundary = np.array([-0.1, 0, 4, 3, 3, 3])
        if np.sum(np.array(logparams<=min_boundary))!=0 or  np.sum(np.array(logparams>=max_boundary))!=0:
                            checkmin=np.array(logparams<=min_boundary)
                            checkmax=np.array(logparams>=max_boundary)
                            logparams=checkmin*(min_boundary+0.0001)+checkmax*(max_boundary-0.0001)+(1-checkmin)*(1-checkmax)*logparams
                            sigma = 10**logparams[0]
                            dt = 10**logparams[1]
                            zeta=10**logparams[2]
                            S=10**logparams[3]
                            D=-10**logparams[4]
                            O=10**logparams[5]

        attempt_num = 2000

        clip_percents=50
        population_size=7
        percentile = 50

        sim=SIMCIM(K, O,S,D, N,dt, tau,  dim, sigma, zeta, alpha, attempt_num,  device='cuda')
        interface = SimcimInterface(sim, attempt_num, percentile)

        first_params = np.array([np.log10(sim.sigma), np.log10(sim.dt), np.log10(sim.zeta), np.log10(sim.S), np.log10(np.abs(sim.D)), np.log10(sim.O)])
        min_boundary = np.array([-3,-4,-4,-4,-4,-4])
        max_boundary = np.array([ -0.1, 0, 4, 3, 3, 3])
        num_params = 6
        config = dict(controller_type='neural_net', 
                      training_type = 'random',
                      first_params = first_params,
                      min_boundary = min_boundary, 
                      max_boundary = max_boundary,
                      num_params = num_params,
                      trust_region = max_boundary - min_boundary,
                      max_num_runs = 1500,
                      num_training_runs = 2 * num_params,
                      log_level = 0, 
                      no_delay = False, 
                      max_num_runs_without_better_params = 350,
                      #evolution_strategy='best1', 
                      population_size=10, 
                      mutation_scale = (0.5,1.0), 
                      cross_over_probability = 0.7, 
                      default_bad_cost = 1.25, 
                      #target_cost=-(8+np.percentile(np.array(sim.lastcuts.cpu()), clip_percents))/(-np.min(np.array(sim.lastcuts.cpu()))+np.percentile(np.array(sim.lastcuts.cpu()), clip_percents)),
                      default_bad_uncertainty = 0.0001,
                      predict_global_minima_at_end = True,
                      cost_has_noise = True)

        controller = mlc.create_controller(interface, **config)
        controller.optimize()
        mlv.show_all_default_visualizations(controller)

        clear_output(True)

        #print('Best cut found:')
        #print(-controller.best_cost*controller.interface.scale+controller.interface.perc)
        sigma=10**controller.best_params[0]
        dt=10**controller.best_params[1]
        zeta=10**controller.best_params[2]
        S=10**controller.best_params[3]
        D=-10**controller.best_params[4]
        O=10**controller.best_params[5]
        attempt_num=2000
        params=np.array([sigma, dt, zeta, S, D, O])
        sim=SIMCIM(K, O,S,D, N,dt, tau,  dim, sigma, zeta, alpha, attempt_num,  device='cuda')
        H_opt, spins_opt =sim.evolve()
        print(H_opt.min())
#         print(np.percentile(np.array(sim.lastcuts.cpu()), 25))

#         print(H_opt.min())
        spins=torch.sign(spins_opt).double()
    
        K_doub=K.double()
        energy=torch.einsum('ijk,is,js,ks->s',(K_doub, spins, spins, spins))
        num = np.argmin(energy.cpu())
        print(energy.cpu().min())
        np.save('PolySimCIM_each/'+str(psize)+'_'+str(number)+'_bit_rand.npy', np.array(spins.cpu())[:, num])
        np.save('PolySimCIM_each/'+str(psize)+'_'+str(number)+'_rand.npy', np.array(energy.cpu()))
        np.save('PolySimCIM_each/'+str(psize)+'_'+str(number)+'_params_rand.npy', params)

##### def pumping(O,S,D,N):
        i = torch.arange(N, dtype = torch.float32)
        sarg = torch.tensor(S,dtype = torch.float32)*(i/N-0.5)
        return O*(torch.tanh(arg) + D )